In [3]:
import pandas as pd
import s3fs
import zipfile

# Configuration pour l'accès public
fs = s3fs.S3FileSystem(anon=True)

bucket_path = 'tripdata'
fichiers_disponibles = fs.ls(bucket_path)
fichiers_zip = [f for f in fichiers_disponibles if f.endswith('.zip')]

print(f"Nombre de fichiers .zip trouvés : {len(fichiers_zip)}")

if len(fichiers_zip) > 0:
    premier_fichier = fichiers_zip[0]
    print(f"Ouverture de l'archive : {premier_fichier}")
    print("-" * 50)
    
    # 1. Ouvrir le flux du fichier S3 (sans le télécharger entièrement)
    with fs.open(premier_fichier, 'rb') as f_s3:
        # 2. Lire l'archive zip en mémoire
        with zipfile.ZipFile(f_s3) as z:
            noms_fichiers = z.namelist()
            
            # 3. Filtrer pour trouver les vrais fichiers CSV (ignorer les dossiers et fichiers cachés Mac)
            vrais_csv = [
                nom for nom in noms_fichiers 
                if nom.endswith('.csv') 
                and not nom.startswith('__MACOSX') 
                and not nom.split('/')[-1].startswith('._')
            ]
            
            if vrais_csv:
                cible_csv = vrais_csv[0]
                print(f"Extraction en mémoire du fichier interne : {cible_csv}...")
                
                # 4. Lire spécifiquement ce CSV avec Pandas
                with z.open(cible_csv) as f_csv:
                    df_citibike = pd.read_csv(f_csv, nrows=5000)
                
                display(df_citibike.head())
                print("-" * 50)
                df_citibike.info()
            else:
                print("Aucun fichier CSV valide n'a été trouvé dans cette archive.")

Nombre de fichiers .zip trouvés : 163
Ouverture de l'archive : tripdata/2013-citibike-tripdata.zip
--------------------------------------------------
Extraction en mémoire du fichier interne : 2013-citibike-tripdata/201309-citibike-tripdata.csv...


,tripduration,starttime,stoptime,start station id,start station name,start station latitude,start station longitude,end station id,end station name,end station latitude,end station longitude,bikeid,usertype,birth year,gender
0,1010,2013-09-01 00:00:02,2013-09-01 00:16:52,254,W 11 St & 6 Ave,40.735324,-73.998004,147,Greenwich St & Warren St,40.715422,-74.011220,15014,Subscriber,1974,1
1,1443,2013-09-01 00:00:09,2013-09-01 00:24:12,151,Cleveland Pl & Spring St,40.721816,-73.997203,497,E 17 St & Broadway,40.737050,-73.990093,19393,Customer,\N,0
2,1387,2013-09-01 00:00:16,2013-09-01 00:23:23,352,W 56 St & 6 Ave,40.763406,-73.977225,405,Washington St & Gansevoort St,40.739323,-74.008119,16160,Subscriber,1992,1
3,405,2013-09-01 00:00:18,2013-09-01 00:07:03,490,8 Ave & W 33 St,40.751551,-73.993934,459,W 20 St & 11 Ave,40.746745,-74.007756,14997,Subscriber,1973,1
4,270,2013-09-01 00:00:20,2013-09-01 00:04:50,236,St Marks Pl & 2 Ave,40.728419,-73.987140,393,E 5 St & Avenue C,40.722992,-73.979955,19609,Subscriber,1984,1


--------------------------------------------------
<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 15 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   tripduration             5000 non-null   int64  
 1   starttime                5000 non-null   str    
 2   stoptime                 5000 non-null   str    
 3   start station id         5000 non-null   int64  
 4   start station name       5000 non-null   str    
 5   start station latitude   5000 non-null   float64
 6   start station longitude  5000 non-null   float64
 7   end station id           5000 non-null   int64  
 8   end station name         5000 non-null   str    
 9   end station latitude     5000 non-null   float64
 10  end station longitude    5000 non-null   float64
 11  bikeid                   5000 non-null   int64  
 12  usertype                 5000 non-null   str    
 13  birth year               5000 non-null

In [ ]:
import pandas as pd
import s3fs
import zipfile

fs = s3fs.S3FileSystem(anon=True)
bucket_path = 'tripdata'
fichiers_disponibles = fs.ls(bucket_path)
fichiers_zip = [f for f in fichiers_disponibles if f.endswith('.zip')]

if len(fichiers_zip) > 0:
    premier_fichier = fichiers_zip[0]
    print(f"Ouverture de l'archive : {premier_fichier}")
    print("-" * 50)
    
    with fs.open(premier_fichier, 'rb') as f_s3:
        with zipfile.ZipFile(f_s3) as z:
            noms_fichiers = z.namelist()
            
            # Filtrage des vrais CSV
            vrais_csv = [
                nom for nom in noms_fichiers 
                if nom.endswith('.csv') 
                and not nom.startswith('__MACOSX') 
                and not nom.split('/')[-1].startswith('._')
            ]
            
            # Liste pour stocker les morceaux de données
            liste_dataframes = []
            
            # On boucle sur TOUS les fichiers CSV trouvés
            for csv_file in vrais_csv:
                print(f"Lecture du fichier interne : {csv_file}...")
                with z.open(csv_file) as f_csv:
                    # On lit l'intégralité du fichier (on enlève le nrows=5000)
                    df_temp = pd.read_csv(f_csv)
                    liste_dataframes.append(df_temp)
            
            # On fusionne tous les tableaux en un seul grand DataFrame
            if liste_dataframes:
                df_citibike_complet = pd.concat(liste_dataframes, ignore_index=True)
                print("-" * 50)
                print(f"Fusion terminée ! Le tableau final contient {len(df_citibike_complet)} lignes.")
                display(df_citibike_complet.head())
            else:
                print("Aucun fichier CSV valide n'a été trouvé.")

Ouverture de l'archive : tripdata/2013-citibike-tripdata.zip
--------------------------------------------------
Lecture du fichier interne : 2013-citibike-tripdata/201309-citibike-tripdata.csv...
